In [1]:
import numpy as np
import pandas as pd
import cv2
import mediapipe as mp
import csv
import os

In [2]:
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python.vision import HandLandmarker, HandLandmarkerOptions, RunningMode

- Thiết lập MediaPipe Hand Landmarker

In [3]:
options = HandLandmarkerOptions(
    base_options=mp_python.BaseOptions(
        model_asset_path='../hand_landmarker.task'
    ),
    running_mode=RunningMode.IMAGE,
    num_hands=1,
    min_hand_detection_confidence=0.7,
    min_tracking_confidence=0.7
)

- Kiểm tra xem camera có nhận được hình ảnh tay không

In [5]:
landmarker = HandLandmarker.create_from_options(options)
cap = cv2.VideoCapture(0)

print("Camera đã bật. Giơ tay vào camera, nhấn Q để thoát.")

while True:
    ret, frame = cap.read()
    frame = cv2.flip(frame, 1)

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = landmarker.detect(mp_image)

    if result.hand_landmarks:
        hand = result.hand_landmarks[0]
        row = []
        for lm in hand:
            row += [lm.x, lm.y, lm.z]

        # Hiển thị số đầu tiên để biết đang chạy đúng
        cv2.putText(frame, f'Thay tay! 63 gia tri, x0={row[0]:.2f}',
                    (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)
    else:
        cv2.putText(frame, 'Khong thay tay',
                    (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)

    cv2.imshow('Thu nghiem', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
landmarker.close()
cv2.destroyAllWindows()


Camera đã bật. Giơ tay vào camera, nhấn Q để thoát.


- Thu thập dữ liệu và lưu trữ

In [4]:
LABELS = [chr(i) for i in range(ord('A'), ord('Z')+1) if chr(i) not in ['J', 'Z']]

SAMPLE_PER_LABEL = 200
OUTPUT_FILE = '../data/data.csv'

landmarker = HandLandmarker.create_from_options(options)
cap = cv2.VideoCapture(0)

with open(OUTPUT_FILE, 'w', newline='') as f:
    writer = csv.writer(f)
    header = ['label'] + [f'{axis}{i}' for i in range(21) for axis in ['x','y','z']]
    writer.writerow(header)

for label in LABELS:
    count = 0

    while True:
        ret, frame = cap.read()
        frame = cv2.flip(frame, 1)
        cv2.putText(frame, f'San sang: {label} | Nhan SPACE', (10,40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,200,255), 2)
        cv2.imshow("Thu thap", frame)

        if cv2.waitKey(1) & 0xFF == ord(' '):
            break
        if cv2.waitKey(1) & 0xFF == ord('q'):
            cap.release()
            cv2.destroyAllWindows() 
            exit()

    while count < SAMPLE_PER_LABEL:
        ret, frame = cap.read()
        frame = cv2.flip(frame, 1)

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        result = landmarker.detect(mp_image)

        if result.hand_landmarks and len(result.hand_landmarks) > 0:
            hand = result.hand_landmarks[0]
            
            row = [label]
            points = []

            h, w, _ = frame.shape  
             
            for lm in hand:
                row += [lm.x, lm.y, lm.z]
                cx, cy = int(lm.x * w), int(lm.y * h)
                points.append((cx, cy))


                cv2.circle(frame, (cx, cy), 4, (0, 255, 0), -1)
            x_list = [int(lm.x * w) for lm in hand]
            y_list = [int(lm.y * h) for lm in hand]

            xmin, xmax = min(x_list), max(x_list)
            ymin, ymax = min(y_list), max(y_list)

            margin = 20
            xmin = max(0, xmin - margin)
            ymin = max(0, ymin - margin)
            xmax = min(w, xmax + margin)
            ymax = min(h, ymax + margin)

            hand_img = frame[ymin:ymax, xmin:xmax]

            if hand_img.size != 0:
                hand_img = cv2.resize(hand_img, (224, 224))

            cv2.rectangle(frame, (xmin, ymin), (xmax, ymax), (0, 255, 255), 2)
            
            connections = [
                (0,1),(1,2),(2,3),(3,4),
                (0,5),(5,6),(6,7),(7,8),
                (0,9),(9,10),(10,11),(11,12),
                (0,13),(13,14),(14,15),(15,16),
                (0,17),(17,18),(18,19),(19,20)
            ]
            
            for start, end in connections:
                cv2.line(frame, points[start], points[end], (255, 0, 0), 2)
            
            img_dir = f'../data/images/{label}'
            os.makedirs(img_dir, exist_ok=True)

            img_path = f'{img_dir}/{count}.jpg'
            cv2.imwrite(img_path, hand_img)
            
            with open(OUTPUT_FILE, 'a', newline='') as f:
                csv.writer(f).writerow(row)
            count+=1

        cv2.putText(frame, f'{label}: {count}/{SAMPLE_PER_LABEL}', (10,40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)
        cv2.imshow("Thu thap", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            cap.release()
            cv2.destroyAllWindows() 
            exit()
            
    print(f'Xong {label}!')
    
cap.release()
cv2.destroyAllWindows()

Xong A!
Xong B!
Xong C!
Xong D!
Xong E!
Xong F!
Xong G!
Xong H!
Xong I!
Xong K!
Xong L!
Xong M!
Xong N!
Xong O!
Xong P!
Xong Q!
Xong R!
Xong S!
Xong T!
Xong U!
Xong V!
Xong W!
Xong X!
Xong Y!
